# Classification d'opinion avec un RNN et un mécanisme d'attention

Vincent Guigue (vincent.guigue@agroparistech.fr)

**Tâche** : classer des critiques de films IMDB en positives / négatives (*sentiment analysis*), cadre *many-to-one* comme pour la classification de noms (4_2), mais avec des **mots** (et des séquences bien plus longues).

**Objectifs**
- utiliser des **embeddings de mots pré-entraînés** (GloVe) ;
- gérer proprement le *padding* de séquences de longueurs très variables ;
- remplacer « le dernier état caché » par une **attention** : une moyenne pondérée de tous les états cachés, dont les poids sont appris... et interprétables.

**Données** (déjà dans `data/`, sinon) :
- critiques IMDB : [https://ai.stanford.edu/~amaas/data/sentiment/](https://ai.stanford.edu/~amaas/data/sentiment/) => `data/aclImdb` ;
- embeddings GloVe : penser à dézipper `data/glove.zip` => `data/glove/glove.6B.50d.txt`.

## Table des matières

- [0. Mise en place : librairies, device, TensorBoard](#0-mise-en-place-librairies-device-tensorboard)
- [A. Chargement des données](#a-chargement-des-donnees)
    - [A.1 Passage au `DataLoader`](#a-1-passage-au-dataloader)
- [B. Création du réseau](#b-creation-du-reseau)
- [C. Apprentissage](#c-apprentissage)
- [D. Analyse des résultats](#d-analyse-des-resultats)
    - [D.1 Comparaison](#d-1-comparaison)
    - [D.2 Visualisation de l'attention](#d-2-visualisation-de-l-attention)
- [E. Pour aller plus loin](#e-pour-aller-plus-loin)

<a id="0-mise-en-place-librairies-device-tensorboard"></a>
## 0. Mise en place : librairies, device, TensorBoard

In [ ]:
import re, time, html
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

In [ ]:
# ---- Choix du device ----
# Pour ces petits RNN, la récurrence est une boucle séquentielle sur le temps : sur Mac, le GPU (MPS)
# est en général PLUS LENT que le CPU (mesuré sur le TP métro : 12 s/epoch en MPS contre 1,2 s/epoch en CPU).
# MPS n'est donc activé que sur demande. Avec un GPU NVIDIA (CUDA), le gain reste réel.
USE_MPS = False

if torch.cuda.is_available():
    device = torch.device("cuda")
elif USE_MPS and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("torch :", torch.__version__, "| device :", device)
_ = torch.manual_seed(0)  # reproductibilité

In [ ]:
# ---- TensorBoard ----
from pathlib import Path
TB_PATH = Path("/tmp/logs/module4")    # une expérience = un sous-répertoire : SummaryWriter(TB_PATH / nom)
try:
    from torch.utils.tensorboard import SummaryWriter
except ImportError:                    # tensorboard absent : on continue sans enregistrer les courbes
    class SummaryWriter:
        def __init__(self, *args, **kwargs): pass
        def add_scalar(self, *args, **kwargs): pass
        def close(self): pass
    print("tensorboard non installé : les courbes ne seront pas enregistrées")

# Option 1 (recommandée) : dans un terminal, lancer la commande ci-dessous puis ouvrir http://localhost:6006
print(f"tensorboard --logdir {TB_PATH}")

# Option 2 : TensorBoard directement dans le notebook (décommenter les 2 lignes)
# %load_ext tensorboard
# %tensorboard --logdir {TB_PATH}

<a id="a-chargement-des-donnees"></a>
## A. Chargement des données

Tout le code est fourni. On ne garde qu'un sous-ensemble des critiques (`train_max_size` par classe) pour que les apprentissages restent rapides.

In [ ]:
GLOVE_PATH   = Path("data/glove")
DATASET_PATH = Path("data/aclImdb")
IMDB_CLASSES = ['neg', 'pos']

class FolderText(Dataset):
    """Dataset basé sur des dossiers (un par classe) et fichiers"""

    def __init__(self, classes, folder: Path, tokenizer, max_size=None):
        self.tokenizer = tokenizer
        self.texts, self.labels = [], []
        for ix, label in enumerate(classes):
            for c, file in enumerate(sorted((folder / label).glob("*.txt"))):
                if max_size is not None and c >= max_size:
                    break
                self.texts.append(file.read_text())
                self.labels.append(ix)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, ix):
        return torch.tensor(self.tokenizer(self.texts[ix])), self.labels[ix]


def get_imdb_data(embedding_size=50, train_max_size=None):
    """Renvoie l'ensemble des données nécessaires pour l'apprentissage
    - liste des mots (id => mot) et dictionnaire mot => id
    - embeddings (GloVe), un par mot ; deux mots spéciaux : __OOV__ (mot inconnu) et __PAD__ (padding)
    - Datasets (FolderText) d'apprentissage et de test
    """
    # un token = une suite de lettres/chiffres OU un signe de ponctuation isolé (comme dans GloVe)
    WORDS = re.compile(r"\w+|[^\w\s]")
    words, embeddings = [], []
    with open(GLOVE_PATH / f"glove.6B.{embedding_size}d.txt", encoding="utf-8") as glove:
        for line in glove:
            values = line.split()
            words.append(values[0])
            embeddings.append([float(x) for x in values[1:]])
    words += ["__OOV__", "__PAD__"]
    embeddings = np.vstack((embeddings, np.zeros((2, embedding_size))))
    word2id = {word: ix for ix, word in enumerate(words)}
    OOVID = word2id["__OOV__"]

    def tokenizer(t):
        t = t.lower().replace("<br />", " ")   # les critiques IMDB contiennent des retours à la ligne HTML
        return [word2id.get(x, OOVID) for x in re.findall(WORDS, t)]

    return (words, word2id, embeddings,
            FolderText(IMDB_CLASSES, DATASET_PATH / "train", tokenizer, train_max_size),
            FolderText(IMDB_CLASSES, DATASET_PATH / "test", tokenizer, train_max_size))

In [ ]:
words, word2id, embeddings, train_dataset, test_dataset = get_imdb_data(train_max_size=1000)
PAD = word2id["__PAD__"]

Prendre le temps de comprendre ce qui est chargé :
- nature des informations ;
- dimension des structures de données ;
- longueur des critiques (en mots).

In [ ]:
print("vocabulaire :", len(word2id), "| embeddings :", embeddings.shape)
print("train :", len(train_dataset), "| test :", len(test_dataset))
x, y = train_dataset[0]
print("exemple 0 : label", y, "|", " ".join(words[i] for i in x[:30]), "...")
lengths = np.array([len(train_dataset[i][0]) for i in range(len(train_dataset))])
oov = sum((train_dataset[i][0] == word2id["__OOV__"]).sum().item() for i in range(len(train_dataset)))
print("taux de mots inconnus (OOV) : %.1f %%" % (100 * oov / lengths.sum()))   # ~16 % avec un découpage naïf sur les espaces !
print("longueurs : moyenne %.0f | médiane %.0f | 90%% %.0f | max %d" % (lengths.mean(), np.median(lengths), np.percentile(lengths, 90), lengths.max()))

<a id="a-1-passage-au-dataloader"></a>
### A.1 Passage au `DataLoader`

Comme pour les noms (4_2), les séquences d'un batch sont complétées avec le mot `__PAD__` et on conserve les longueurs réelles. Les critiques sont de plus **tronquées** à `MAX_LEN` mots : quelques critiques très longues (> 1000 mots) obligeraient sinon à padder tout leur batch, et un RNN sur 1000 pas de temps est lent... et peine à propager l'information.

In [ ]:
BATCH_SIZE = 100
MAX_LEN = 400

def collate_fn(batch):
    sequences, labels = zip(*batch)
    sequences = [seq[:MAX_LEN] for seq in sequences]
    lengths = torch.tensor([len(seq) for seq in sequences])
    padded_sequences = pad_sequence(sequences, batch_first=False, padding_value=PAD)   # Lmax x batch
    return padded_sequences, lengths, torch.tensor(labels)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

x, lengths, y = next(iter(train_loader))
print("x:", x.size(), "| lengths:", lengths[:10], "| y:", y[:10])

<a id="b-creation-du-reseau"></a>
## B. Création du réseau

Le réseau a deux modes, selon l'argument `attention` :

1. **Sans attention** : la représentation de la critique est le dernier état caché (le vrai, pas celui de la dernière position paddée : utiliser `pack_padded_sequence`, cf. 4_2).
2. **Avec attention** : la représentation est une moyenne pondérée de **tous** les états cachés $h_1, \dots, h_T$ :
$$ s_t = w^\top h_t, \qquad \alpha_t = \frac{e^{s_t}}{\sum_{t'} e^{s_{t'}}}, \qquad z = \sum_t \alpha_t h_t$$
   Le vecteur $w$ est appris. Attention au *padding* : les positions paddées ne doivent recevoir **aucun** poids => on met leur score à $-\infty$ avant le softmax, grâce à un **masque**.

Étapes :
- **embeddings** : initialiser la couche `nn.Embedding` avec les vecteurs GloVe (attention au type : numpy est en `float64`, PyTorch attend du `float32`). Faut-il apprendre ces embeddings ? Sinon, comment désactiver le gradient ?
- **attention** : quel module utiliser pour calculer les scores $s_t$ ?
- **masque** : à comprendre impérativement (afficher le masque d'un petit batch et sa taille).

In [ ]:
class RNNSent(nn.Module):
    def __init__(self, hidden_size, output_size, embeddings, cell="rnn", attention=False):
        super().__init__()
        self.cell, self.use_attention = cell, attention

        # INITIALISATION des embeddings
        # 1. Récupération des valeurs présentes dans la structure (numpy) embeddings
        # 2. Faut-il activer le gradient sur ce module ? Dans la négative, comment le désactiver ?
        ##  TODO 
        input_size = self.emb.embedding_dim

        # CHOIX du module récurrent
        self.rec = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[cell](input_size, hidden_size)

        # ATTENTION : réfléchir à la nature du module à utiliser pour calculer les scores s_t
        ##  TODO 

        self.h2o = nn.Linear(hidden_size, output_size)

    def forward(self, input, lengths):
        """ input : Lmax x batch ; lengths : longueurs réelles
            => logits : batch x output_size, poids d'attention : Lmax x batch (None sans attention) """
        # 1. indices => embeddings
        # 2. passage dans le module récurrent (pack_padded_sequence)
        xemb = self.emb(input)
        packed = pack_padded_sequence(xemb, lengths.cpu(), enforce_sorted=False)
        out, last = self.rec(packed)
        if self.cell == "lstm":
            last = last[0]                         # (h_n, c_n) => h_n

        if not self.use_attention:                 # dernier (vrai) état caché
            return self.h2o(last[-1]), None

        # 3. Attention
        hidden, _ = pad_packed_sequence(out)       # Lmax x batch x hidden (0 aux positions paddées)
        maxlen, batch_size = hidden.size(0), hidden.size(1)
        # 3.1 scores d'attention s_t : Lmax x batch
        ##  TODO 
        # 3.2 masque : True pour les vraies positions, False pour le padding (Lmax x batch)
        mask = torch.arange(maxlen, device=hidden.device).unsqueeze(1) < lengths.to(hidden.device).unsqueeze(0)
        scores = scores.masked_fill(~mask, float('-inf'))
        # 3.3 poids (softmax sur le temps) + moyenne pondérée des états cachés
        ##  TODO 
        return self.h2o(z), alpha

In [ ]:
# Compréhension du masque sur un exemple jouet : 3 séquences de longueurs 2, 4 et 3
toy_lengths = torch.tensor([2, 4, 3])
toy_mask = torch.arange(4).unsqueeze(1) < toy_lengths.unsqueeze(0)
print(toy_mask)   # Lmax x batch

# Vérification des dimensions sur un batch, avec et sans attention
loss = nn.CrossEntropyLoss()
x, lengths, y = next(iter(train_loader))
for att in [False, True]:
    model = RNNSent(64, 2, embeddings, attention=att)
    yhat, alpha = model(x, lengths)
    print(f"attention={att} : yhat {yhat.size()} | loss {loss(yhat, y).item():.3f}",
          "" if alpha is None else f"| alpha {alpha.size()}, somme sur le temps : {alpha.sum(0)[:3]}")

<a id="c-apprentissage"></a>
## C. Apprentissage

Boucle standard (identique à 4_2). Apprendre deux modèles : sans puis avec attention ; comparer les courbes dans TensorBoard.

In [ ]:
def evaluate(model, loader):
    model.eval()
    cumloss, cumacc, count = 0., 0, 0
    with torch.no_grad():
        for x, lengths, y in loader:
            x, y = x.to(device), y.to(device)
            yhat, _ = model(x, lengths)
            cumloss += loss(yhat, y).item() * len(y)
            cumacc += (yhat.argmax(1) == y).sum().item()
            count += len(y)
    return cumloss / count, cumacc / count


def train(model, epochs, train_loader, test_loader, name):
    writer = SummaryWriter(TB_PATH / f"{name}-{time.strftime('%Y%m%d-%H%M%S')}")
    model = model.to(device)
    optim = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in tqdm(range(epochs), desc=name):
        model.train()
        cumloss, cumacc, count = 0., 0, 0
        for x, lengths, y in train_loader:
            x, y = x.to(device), y.to(device)
            optim.zero_grad()
            yhat, _ = model(x, lengths)
            l = loss(yhat, y)
            l.backward()
            optim.step()
            cumloss += l.item() * len(y)
            cumacc += (yhat.argmax(1) == y).sum().item()
            count += len(y)
        writer.add_scalar("loss/train", cumloss / count, epoch)
        writer.add_scalar("accuracy/train", cumacc / count, epoch)
        l, acc = evaluate(model, test_loader)
        writer.add_scalar("loss/test", l, epoch)
        writer.add_scalar("accuracy/test", acc, epoch)
    writer.close()
    return model

In [ ]:
# ~1 min par modèle sur CPU
n_hidden, n_epoch = 64, 15
rnn_last = train(RNNSent(n_hidden, 2, embeddings, attention=False), n_epoch, train_loader, test_loader, "imdb-rnn-last")
rnn_attn = train(RNNSent(n_hidden, 2, embeddings, attention=True),  n_epoch, train_loader, test_loader, "imdb-rnn-attention")

<a id="d-analyse-des-resultats"></a>
## D. Analyse des résultats

<a id="d-1-comparaison"></a>
### D.1 Comparaison

Comparer les performances en test des deux modèles. Comment expliquer la différence ?

In [ ]:
for name, model in [("dernier état caché", rnn_last), ("attention", rnn_attn)]:
    print(f"{name:20s} : test loss %.3f | accuracy %.3f" % evaluate(model, test_loader))
##  TODO 

<a id="d-2-visualisation-de-l-attention"></a>
### D.2 Visualisation de l'attention

Les poids $\alpha_t$ indiquent quels mots ont contribué à la décision. Afficher une critique du test en surlignant chaque mot proportionnellement à son poids d'attention. Les mots mis en avant sont-ils pertinents ?

In [ ]:
from IPython.display import HTML, display

def show_attention(model, idx):
    x, y = test_dataset[idx]
    x = x[:MAX_LEN]
    model.eval()
    with torch.no_grad():
        yhat, alpha = model(x.unsqueeze(1).to(device), torch.tensor([len(x)]))
    alpha = alpha[:, 0].cpu()
    ##  TODO 

for idx in [0, 1500]:   # une critique négative, une positive
    show_attention(rnn_attn, idx)

<a id="e-pour-aller-plus-loin"></a>
## E. Pour aller plus loin

- Remplacer le RNN par un LSTM ou un GRU (argument `cell`) : l'écart entre les deux modes se réduit-il ?
- Ré-apprendre les embeddings (`freeze=False`) : gain ou sur-apprentissage ?
- Utiliser toutes les données (25 000 critiques) et des embeddings de plus grande dimension (100, 300).
- Attention multi-têtes : plusieurs vecteurs $w$ => plusieurs moyennes pondérées concaténées. On se rapproche des Transformers (module 5) !